# How Learning Can Guide Evolution Recreation

The following notebook is an attempt at recreating Hinton and Nowlan's paper that proves the relevance of the Baldwin effect with a simple example, using alleles with 20 genes. Said genes have three possible states 1, for the existence of said trait, 0, for the lack thereof, or ? for the possibility of being open to learning or guessing during the organism's lifetime. 

If the organism has a ? gene it should receive multiple lifetime trials to guess/learn the correct setting for the open slots containing the sign. If it figures out the correct configuration, it gets rewarded with high fitness. What it learns is not written back into the genes. 

The idea here is that the learning gives a survival advantage to organisms that have **some** of the correct genes preset, and attribute the remaining work to learning. 
Instead of a flat landscape where only the perfect genotype survives, learning should introduce the possibility for organisms with a good genettic start to learn the rest quickly and survive. 
Over generations natiral selection should favor genotypes with the correct 1s because it reduces the cost of learning from scratch. Eventually the hardcoded ? becomes 1. 

``The "training a few epochs" part isn't the mechanism that resolves ? — it's just the unavoidable cost of finding out whether this particular random guess was any good, since (unlike the original paper's instant bit-comparison) you can't know if a hyperparameter combo is good without actually running it. If it doesn't clear the threshold, throw that whole trained network away and try again with a fresh random draw of the ? bits — don't continue training the same network, don't nudge it toward the previous guess. Each trial is a fully independent roll of the dice, exactly like the original.``

One organism's lifetime, end to end

Genotype fixes some bits, leaves others ?.

For trial = 1 to max_trials

Randomly resample all ? bits.

Decode the complete bit string → architecture + hyperparameters.

Build and briefly train that NN from scratch.

Evaluate on held-out validation data. Clears threshold? → record trial number, stop lifetime, done.

Doesn't clear it? → discard, next trial.

If no trial succeeds within the budget → treat as failure, low fitness.

Fitness = increasing function of "cleared the threshold" and decreasing function of "how many trials it took" (or a fixed low value if it never succeeded).

## OPTIONS FOR HYPERPARAMETERS

| Hyperparameter |	Bits |	Menu (decoded by bit value) |
| -------- | ------- | ------- |
| n_hidden_layers	| 2	| 1, 2, 3, 4 |
| units_per_layer	| 3	| 8, 16, 32, 64, 128, 256, 512, 1024 |
| activation	| 1	| relu, tanh |
| learning_rate	| 2	| 0.1, 0.01, 0.001, 0.0001 |
| batch_size	| 2	| 16, 32, 64, 128 |
| dropout_rate	| 2	| 0.0, 0.1, 0.2, 0.3 |

## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pandas.plotting import scatter_matrix

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import TargetEncoder

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression

from sklearn.compose import TransformedTargetRegressor

from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold
from sklearn.model_selection import ShuffleSplit

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.dummy import DummyRegressor

from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_absolute_error

In [2]:
def plot_keras_history(history, metric):
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    fig.tight_layout()
    axes[0].plot(history.history["loss"], label="train loss")
    axes[0].plot(history.history["val_loss"], label="val loss")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[1].plot(history.history[metric], label="train " + metric)
    axes[1].plot(history.history["val_" + metric], label="val " + metric)
    axes[1].set_title(metric)
    axes[1].legend()
    plt.show()

Below we specify the possible genes that can will be used to define the NN. Each hyperparameter has a variable amounts of genes that represent it, according to the possible options for it (eg hidden layer only requires 2 bits whilst units per layer requires 3)

A genome is a sequence of bits where each bit has possible values

1 -> fixed bit

0 -> fixed bit

None -> ? gene, unresolved and guessed on each trial.


The sequence of bits will be used to decode and encode the options for the hyperparameter.

In [4]:
GENE_SPEC = [
    ("n_hidden_layers", 2, [1, 2, 3, 4]),
    ("units_per_layer", 3, [8, 16, 32, 64, 128, 256, 512, 1024]),
    ("activation", "relu", ["relu", "tanh"]),
    ("learning_rate", 2, [0.001, 0.01, 0.1]),
    ("batch_size", 2, [16, 32, 64]),
    ("alpha", 2, [0.0001, 0.001, 0.01, 0.1])
]

GENOME_LENGTH = sum(bits for _, bits, _ in GENE_SPEC) #12 bits


KeyboardInterrupt



In [ ]:
def randomize_genome(p_learn=0.3):
    """Randomly generate a genome with equal probabilities of 0 and 1, 
    and some bits missing (None - ?) based on the probability of learning"""
    rng = np.random.default_rng()
    genome = []
    for _ in range(GENOME_LENGTH):
        if rng.random() < p_learn:
            genome.append(None)
        else:
            genome.append(rng.integers(0, 2))
    return genome

In [ ]:
def resolve(genome):
    """One lifetime worth of guessing the missing bits in a genome. 
    Fixed bits are preserved, while missing bits are randomly guessed.
    Returns a conrete bit list without any Nones which will then be decoded""" 
    rng = np.random.default_rng()
    return [int(rng.integers(0,2)) if bit is None else bit for bit in genome]

In [ ]:
def decode(bits):
    """Return a conrete hyperparameter dictionary from a randomized concrete bit list (all 0/1)"""
    hp = {}
    pos = 0 
    for name, n_bits, menu in GENE_SPEC:
        segment = bits[pos:pos+n_bits]
        pos += n_bits
        index = int("".join(str(b) for b in segment), 2) # binary to int
        hp[name] = menu[index]
    return hp 

In [ ]:
def visualize_genome(genome):
    """Visualize a genome as a string of 0/1/N"""
    return " ".join("?" if b is None else str(b) for b in genome)

In [7]:

import os
base_dir = ".."
dataset_dir = os.path.join(base_dir, "datasets")
df = pd.read_csv(os.path.join(dataset_dir, "diamonds.csv"))